# Tool Definition Caching — InvokeModel API (Anthropic)

This notebook demonstrates how to cache tool (function) definitions using the **InvokeModel** and **InvokeModelWithResponseStream** APIs with Anthropic Claude's native `cache_control` syntax.

## What you'll learn

- **InvokeModel API**: `cache_control` on the last tool definition
- **InvokeModelWithResponseStream**: Same syntax with streaming response parsing
- Schema format differences vs Converse API

## Anthropic-specific tool cache placement

```python
"tools": [
    {"name": ..., "input_schema": ...},
    {"name": ..., "input_schema": ..., "cache_control": {"type": "ephemeral", "ttl": "5m"}}
]
```

For the model-agnostic Converse API syntax (`cachePoint` appended after all tools), see [converse_api/](../../converse_api/).

## Prerequisites

- AWS credentials configured (default profile)
- Access to Claude Sonnet 4.6 on Amazon Bedrock
- `boto3 >= 1.42.0`

## Token threshold

Claude Sonnet 4.6 requires at least **2,048 tokens** per cache checkpoint. The 4 tool definitions below collectively exceed this threshold.

In [ ]:
!pip install --upgrade --quiet boto3

In [ ]:
import boto3
import json
import time

MODEL_ID = "global.anthropic.claude-sonnet-4-6"
AWS_REGION = "us-west-2"
CACHE_TTL = "5m"

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)
print(f"Model: {MODEL_ID}")

## Tool Definitions

Space-themed tools with comprehensive schemas.

In [ ]:
TOOLS_LIST = [
    {
        "name": "analyze_celestial_object",
        "description": "Analyzes a celestial object and returns detailed information about its physical properties, composition, and characteristics. Use this tool when you need comprehensive data about planets, moons, stars, asteroids, or other celestial bodies.",
        "input_schema": {
            "type": "object",
            "properties": {
                "object_name": {"type": "string", "description": "The name or designation of the celestial object (e.g., 'Mars', 'Europa', 'Proxima Centauri b')"},
                "object_type": {"type": "string", "enum": ["planet", "dwarf_planet", "moon", "asteroid", "comet", "star", "exoplanet", "nebula", "galaxy"], "description": "The classification of the celestial object"},
                "analysis_depth": {"type": "string", "enum": ["basic", "standard", "comprehensive"], "description": "Level of detail for the analysis"},
                "include_moons": {"type": "boolean", "description": "Whether to include information about the object's natural satellites"},
                "coordinate_system": {"type": "string", "enum": ["ecliptic", "equatorial", "galactic"], "description": "The coordinate system to use for positional data"},
                "epoch": {"type": "string", "description": "The reference epoch for orbital elements (e.g., 'J2000')"},
                "output_units": {
                    "type": "object",
                    "properties": {
                        "mass": {"type": "string", "enum": ["kg", "earth_masses", "solar_masses", "jupiter_masses"]},
                        "distance": {"type": "string", "enum": ["km", "au", "light_years", "parsecs"]},
                        "temperature": {"type": "string", "enum": ["kelvin", "celsius", "fahrenheit"]}
                    }
                }
            },
            "required": ["object_name", "object_type"]
        }
    },
    {
        "name": "calculate_orbital_mechanics",
        "description": "Performs orbital mechanics calculations including trajectory planning, orbital transfers, gravitational interactions, and mission feasibility assessments. Supports both two-body and n-body calculations.",
        "input_schema": {
            "type": "object",
            "properties": {
                "calculation_type": {"type": "string", "enum": ["hohmann_transfer", "gravity_assist", "orbital_period", "escape_velocity", "delta_v", "sphere_of_influence", "lagrange_points", "orbital_decay"]},
                "origin_body": {"type": "string", "description": "The starting celestial body or orbit"},
                "destination_body": {"type": "string", "description": "The target celestial body or orbit"},
                "orbital_elements": {
                    "type": "object",
                    "properties": {
                        "semi_major_axis": {"type": "number"},
                        "eccentricity": {"type": "number"},
                        "inclination": {"type": "number"},
                        "longitude_ascending_node": {"type": "number"},
                        "argument_periapsis": {"type": "number"},
                        "true_anomaly": {"type": "number"}
                    }
                },
                "spacecraft_mass": {"type": "number", "description": "Mass of the spacecraft in kilograms"},
                "propulsion_system": {
                    "type": "object",
                    "properties": {
                        "type": {"type": "string", "enum": ["chemical", "ion", "nuclear_thermal", "solar_sail"]},
                        "specific_impulse": {"type": "number"},
                        "thrust": {"type": "number"}
                    }
                },
                "launch_window": {
                    "type": "object",
                    "properties": {
                        "start_date": {"type": "string"},
                        "end_date": {"type": "string"},
                        "optimization_criteria": {"type": "string", "enum": ["minimum_delta_v", "minimum_time", "balanced"]}
                    }
                }
            },
            "required": ["calculation_type", "origin_body"]
        }
    },
    {
        "name": "search_exoplanet_database",
        "description": "Searches comprehensive exoplanet databases including NASA Exoplanet Archive and EU Exoplanet Catalogue. Returns detailed information about confirmed exoplanets, host stars, detection methods, and habitability assessments.",
        "input_schema": {
            "type": "object",
            "properties": {
                "search_mode": {"type": "string", "enum": ["by_name", "by_star", "by_criteria", "by_region"]},
                "planet_name": {"type": "string"},
                "host_star": {"type": "string"},
                "discovery_method": {"type": "string", "enum": ["transit", "radial_velocity", "direct_imaging", "microlensing", "timing", "astrometry"]},
                "mass_range": {"type": "object", "properties": {"min": {"type": "number"}, "max": {"type": "number"}}},
                "radius_range": {"type": "object", "properties": {"min": {"type": "number"}, "max": {"type": "number"}}},
                "orbital_period_range": {"type": "object", "properties": {"min": {"type": "number"}, "max": {"type": "number"}}},
                "habitable_zone_only": {"type": "boolean"},
                "equilibrium_temperature_range": {"type": "object", "properties": {"min": {"type": "number"}, "max": {"type": "number"}}},
                "stellar_type": {"type": "string", "enum": ["O", "B", "A", "F", "G", "K", "M"]},
                "distance_from_earth": {"type": "object", "properties": {"max_light_years": {"type": "number"}}},
                "sort_by": {"type": "string", "enum": ["distance", "mass", "radius", "discovery_date", "habitability_score"]},
                "max_results": {"type": "integer"}
            },
            "required": ["search_mode"]
        }
    },
    {
        "name": "simulate_mission_trajectory",
        "description": "Simulates complete spacecraft mission trajectories from launch to destination arrival. Includes multi-body gravitational effects, maneuver planning, and mission phase modeling.",
        "input_schema": {
            "type": "object",
            "properties": {
                "mission_name": {"type": "string"},
                "mission_type": {"type": "string", "enum": ["flyby", "orbiter", "lander", "sample_return", "crewed", "telescope"]},
                "launch_site": {"type": "string", "enum": ["kennedy_space_center", "vandenberg", "baikonur", "kourou", "tanegashima", "satish_dhawan", "cape_canaveral"]},
                "launch_date": {"type": "string"},
                "target_body": {"type": "string"},
                "gravity_assists": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "body": {"type": "string"},
                            "approach_altitude": {"type": "number"},
                            "expected_date": {"type": "string"}
                        }
                    }
                },
                "spacecraft_configuration": {
                    "type": "object",
                    "properties": {
                        "dry_mass": {"type": "number"},
                        "fuel_mass": {"type": "number"},
                        "power_system": {"type": "string", "enum": ["solar_panels", "rtg", "nuclear_reactor", "battery"]},
                        "communication_band": {"type": "string", "enum": ["S_band", "X_band", "Ka_band", "optical"]}
                    }
                },
                "simulation_parameters": {
                    "type": "object",
                    "properties": {
                        "time_step": {"type": "number"},
                        "include_perturbations": {"type": "boolean"},
                        "include_solar_pressure": {"type": "boolean"},
                        "monte_carlo_runs": {"type": "integer"}
                    }
                },
                "output_format": {"type": "string", "enum": ["summary", "detailed", "trajectory_data", "visualization"]}
            },
            "required": ["mission_name", "mission_type", "launch_date", "target_body"]
        }
    }
]

QUESTION = "I want to plan a mission to Europa. Can you help me understand if it's feasible and what tools you'd use?"

print(f"Defined {len(TOOLS_LIST)} tools")

## 1. InvokeModel — Tool Definition Caching

Tools use `input_schema` directly (no `json` wrapper). The `cache_control` is added **inside the last tool definition**:

```python
"tools": [
    {"name": ..., "input_schema": ...},
    {"name": ..., "input_schema": ..., "cache_control": {"type": "ephemeral", "ttl": "5m"}}
]
```

In [ ]:
def build_invoke_model_tools():
    tools = []
    for i, tool in enumerate(TOOLS_LIST):
        entry = {
            "name": tool["name"],
            "description": tool["description"],
            "input_schema": tool["input_schema"]
        }
        if i == len(TOOLS_LIST) - 1:
            entry["cache_control"] = {"type": "ephemeral", "ttl": CACHE_TTL}
        tools.append(entry)
    return tools


def invoke_model_tools_cached():
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 512,
        "tools": build_invoke_model_tools(),
        "messages": [{"role": "user", "content": QUESTION}]
    }

    response = bedrock.invoke_model(modelId=MODEL_ID, body=json.dumps(request_body))
    return json.loads(response["body"].read()).get("usage", {})

# Request 1 — cache write
usage1 = invoke_model_tools_cached()
print("Request 1 (cache write expected):")
print(json.dumps(usage1, indent=2))

time.sleep(1)

# Request 2 — cache read
usage2 = invoke_model_tools_cached()
print("\nRequest 2 (cache read expected):")
print(json.dumps(usage2, indent=2))

## 2. InvokeModelWithResponseStream — Tool Definition Caching with Streaming

Same tool caching pattern with streaming output.

In [ ]:
def invoke_model_stream_tools_cached():
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 512,
        "tools": build_invoke_model_tools(),
        "messages": [{"role": "user", "content": QUESTION}]
    }

    response = bedrock.invoke_model_with_response_stream(
        modelId=MODEL_ID,
        body=json.dumps(request_body)
    )

    text = ""
    usage_data = {}
    for event in response["body"]:
        chunk = json.loads(event["chunk"]["bytes"])
        if chunk["type"] == "content_block_delta":
            text += chunk["delta"].get("text", "")
        elif chunk["type"] == "message_delta":
            if "usage" in chunk:
                usage_data.update(chunk["usage"])
        elif chunk["type"] == "message_start":
            if "message" in chunk and "usage" in chunk["message"]:
                usage_data.update(chunk["message"]["usage"])

    return usage_data

# Request 1 — cache write
usage1 = invoke_model_stream_tools_cached()
print("Request 1 (cache write expected):")
print(json.dumps(usage1, indent=2))

time.sleep(1)

# Request 2 — cache read
usage2 = invoke_model_stream_tools_cached()
print("\nRequest 2 (cache read expected):")
print(json.dumps(usage2, indent=2))

## Schema Format Comparison

| Feature | InvokeModel API (Anthropic) | Converse API |
|---|---|---|
| Schema wrapper | `input_schema: {...}` (no wrapper) | `inputSchema: {"json": {...}}` |
| Cache placement | `"cache_control"` added inside last tool object | `{"cachePoint": ...}` appended as separate array element |
| Tool wrapper | Direct object `{"name": ..., "input_schema": ...}` | `{"toolSpec": {...}}` |
| Usage keys | `cache_creation_input_tokens`, `cache_read_input_tokens` | `cacheWriteInputTokens`, `cacheReadInputTokens` |

## Conclusion

Tool definition caching with InvokeModel uses `cache_control` on the **last tool** in the `tools` array. This caches all preceding tools as well.

For the model-agnostic Converse API approach (recommended for new applications), see [converse_api/](../../converse_api/).

For more details, see the [Amazon Bedrock Prompt Caching documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html).